# Première boucle : apprendre sur trois exemples fixes

Assembler les opérations déjà comprises avant d’ajouter Dataset ou padding.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#premiere-boucle-minimale).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous revenons à nos trois extraits fictifs et leurs deux mesures. La cible est toujours voix, percussion, voix. Les premiers poids reproduisent la mauvaise préférence pour B.

Le tableau est assez petit pour être donné directement au modèle. Nous n’ajoutons ni fichiers, ni chargeur, ni séquences de durées différentes.

modele.parameters() fournit les poids et biais enregistrés. L’optimiseur les règle ensemble. Une itération traite ici le lot entier : elle correspond donc à une époque sur ce minuscule jeu.

range(201) produit les indices 0 à 200, soit 201 itérations. Les pertes imprimées sont celles de la passe avant, juste avant chaque pas.

torch.set_num_threads(1) demande à PyTorch d’utiliser un seul fil d’exécution, ou thread, pour ses calculs CPU. C’est un réglage d’exécution, sans rôle dans la règle apprise.

torch.manual_seed(7) fixe le point de départ des tirages pseudoaléatoires de PyTorch. Nous choisissons ici 7 ; une même graine redonne la même suite de tirages dans le même environnement.

Linear initialise normalement ses poids par tirage. Ici, nous remplaçons aussitôt ses poids et biais à la main. Le départ effectif de l’apprentissage reste donc le tableau de poids déjà étudié.

## Démonstration 1 · Réunir données, couche, perte et SGD

Reconnaître une boucle faite de briques déjà étudiées.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Un seul thread suffit. La graine rend reproductibles les éventuels tirages, même si nous remplaçons ici les premiers poids à la main.

In [4]:
torch.set_num_threads(1)
torch.manual_seed(7)
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]], dtype=torch.float32)
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [5]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

Linear(2,2) transforme les deux mesures en deux scores. Les poids et biais choisis sont identiques à ceux du chapitre Linear.

In [6]:
modele = torch.nn.Linear(2, 2)


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [7]:
with torch.no_grad():
    modele.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.bias.zero_()


### Choisir les paramètres et leur mise à jour

Le critère compare scores et annotations. L’optimiseur reçoit les paramètres du modèle et le taux de 0,1.

In [8]:
critere = torch.nn.CrossEntropyLoss()
optimiseur = torch.optim.SGD(modele.parameters(), lr=0.1)


### Calculer l’état suivant

La liste historique garde des nombres Python pour observer la progression. Chaque tour de for travaille sur les mêmes trois exemples.

In [9]:
historique = []


In [10]:
observer('historique', historique)

historique = []


### Répéter la même étape sur les données

zero_grad efface les anciennes contributions. modele(X) recalcule les scores courants ; le critère produit une perte scalaire.

backward et step jouent leurs deux rôles distincts. Nous ne prenons pas de décision argmax avant le calcul de la perte.

item() fournit un nombre détaché de la mécanique de dérivation pour le journal. Le if limite les impressions à trois tours.

In [11]:
for epoque in range(201):
    optimiseur.zero_grad()
    scores = modele(X)
    perte = critere(scores, cibles)
    perte.backward()
    optimiseur.step()
    historique.append(perte.item())
    if epoque in [0, 50, 200]:
        print("epoque", epoque, "perte avant pas", round(perte.item(), 6))


epoque 0 perte avant pas 3.018476
epoque 50 perte avant pas 0.514087
epoque 200 perte avant pas 0.226087


### Appliquer le contexte explicite

La dernière passe ne demande pas de gradient. Nous calculons les décisions et un coût nouveau après le dernier pas.

In [12]:
with torch.no_grad():
    scores_finaux = modele(X)
    predictions = scores_finaux.argmax(dim=1)
    perte_finale = critere(scores_finaux, cibles)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [13]:
print("decisions finales", predictions.tolist())
print("cibles", cibles.tolist(), "perte finale", round(perte_finale.item(), 6))

decisions finales [0, 1, 0]
cibles [0, 1, 0] perte finale 0.225241


Les trois décisions finales coïncident avec les trois annotations. Ce résultat montre la mécanique de la boucle ; il ne mesure aucune généralisation à de nouveaux sons.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Diagnostiquez deux itérations sur w² sans zero_grad : quel poids final depuis w=1, avec lr=0,1 ?

In [14]:
exo_w = torch.tensor(1.,requires_grad=True)
exo_opt = torch.optim.SGD([exo_w],lr=.1)
for _ in range(2):
    (exo_w**2).backward()
    exo_opt.step()
exo_poids = ...  # Remplacer les points par votre réponse.
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Le premier gradient vaut 2. Le second, 1,6, s’ajoute au précédent.

</details>

### Exercice 2

Nous remettons cette fois les gradients à zéro au début de chaque tour. Prévoyez le poids final.

In [15]:
exo_w = torch.tensor(1.,requires_grad=True)
exo_opt = torch.optim.SGD([exo_w],lr=.1)
for _ in range(2):
    exo_opt.zero_grad()
    (exo_w**2).backward()
    exo_opt.step()
exo_poids = ...  # Remplacer les points par votre réponse.
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le premier poids devient 0,8. Le second pas utilise seulement le gradient courant, 1,6.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_w = torch.tensor(1.,requires_grad=True)
exo_opt = torch.optim.SGD([exo_w],lr=.1)
for _ in range(2):
    (exo_w**2).backward()
    exo_opt.step()
exo_poids = 0.44
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_w = torch.tensor(1.,requires_grad=True)
exo_opt = torch.optim.SGD([exo_w],lr=.1)
for _ in range(2):
    exo_opt.zero_grad()
    (exo_w**2).backward()
    exo_opt.step()
exo_poids = 0.64
if exo_poids is Ellipsis or exo_poids is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_poids)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [16]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()